# Ajuste eficiente: una parte pequeña, un modelo grande

**Materiales desarrollados por Matías Barreto, 2026**

**Tecnicatura Superior en Ciencias de Datos e IA, IFTS N.º 24**
* **Nomenclatura oficial:** Procesamiento de Lenguaje Natural
* **Nombre de trabajo:** Laboratorio de PLN: Analítica, Textos y Cultura

---

| | |
|---|---|
| **Problema** | Un modelo que escribe texto sabe mucho, pero no sabe en qué formato querés la respuesta. |
| **Producto** | Un adaptador LoRA guardado y una medición antes y después, con los mismos ejemplos. |
| **Recursos** | Google Colab con GPU. El modelo base descarga unos 2,2 GB. |

Una oficina recibe avisos breves y necesita que cada uno vuelva como un JSON con dos claves, `tema` y `síntesis`. Un modelo instructivo lo intenta, pero a veces agrega explicaciones o inventa claves.

**¿Puede un modelo aprender ese formato ajustando solo una parte pequeña de sus pesos?**

> **Extensión opcional.** No forma parte del recorrido obligatorio: no penaliza a quien no pueda ejecutarla, y se puede leer y discutir con los resultados de una demostración docente.
>
> **Estado: sin validar.** Este cuaderno todavía no se ejecutó de punta a punta en Colab con GPU. Unsloth y TRL cambian seguido: si una celda falla por un argumento que ya no existe, revisá la documentación de la biblioteca antes de tocar el resto. Lo que se guarda en Colab se borra al cerrar la sesión.

## 1. Entorno y GPU

Elegí `Entorno de ejecución > Cambiar tipo de entorno > GPU` antes de empezar. Este cuaderno no usa el entorno de `uv` ni corre en una Mac, y no necesita otros archivos: los datos viajan adentro.

**Unsloth** es una biblioteca que acelera el ajuste de modelos con LoRA. Las versiones de `transformers` y `trl` que se instalan abajo son las que fijaba el cuaderno oficial de Unsloth para Llama 3.2 al 22 de octubre de 2026; no las probamos acá.

In [ ]:
%%capture
# %%capture oculta la salida de la instalacion, que es muy larga.
!pip install unsloth
!pip install transformers==4.57.6
!pip install --no-deps trl==0.22.2

In [ ]:
import json

import torch

assert torch.cuda.is_available(), (
    "No hay GPU. Elegi Entorno de ejecucion > Cambiar tipo de entorno > "
    "GPU y volve a empezar."
)
print("GPU:", torch.cuda.get_device_name(0))

## 2. Los datos

Cada ejemplo tiene una instrucción, un aviso y la respuesta esperada: un JSON con `tema` (educación, movilidad o salud) y `síntesis`. Son 36 avisos sintéticos (CC0), separados desde el comienzo en `train` (para aprender), `val` (para medir antes y después) y `test` (una comprobación final, que se abre una sola vez).

Son pocos y simples: sirven para observar el procedimiento, no para estimar rendimiento real. Que la respuesta sea un JSON válido no garantiza que el contenido sea correcto.

In [ ]:
DATOS_EMBEBIDOS = r'''
[
  {"id": "ficha_0", "instruccion": "Convertí el aviso en JSON con las claves tema y síntesis. Tema debe ser educación, movilidad o salud. No agregues información.", "entrada": "Abrió la inscripción a talleres de escritura en la biblioteca barrial.", "respuesta": "{\"tema\": \"educación\", \"síntesis\": \"Abrió la inscripción a talleres de escritura en la biblioteca barrial.\"}", "split": "train"},
  {"id": "ficha_1", "instruccion": "Convertí el aviso en JSON con las claves tema y síntesis. Tema debe ser educación, movilidad o salud. No agregues información.", "entrada": "La escuela organiza una muestra de proyectos de ciencias.", "respuesta": "{\"tema\": \"educación\", \"síntesis\": \"La escuela organiza una muestra de proyectos de ciencias.\"}", "split": "train"},
  {"id": "ficha_2", "instruccion": "Convertí el aviso en JSON con las claves tema y síntesis. Tema debe ser educación, movilidad o salud. No agregues información.", "entrada": "El instituto ofrece tutorías para preparar los exámenes.", "respuesta": "{\"tema\": \"educación\", \"síntesis\": \"El instituto ofrece tutorías para preparar los exámenes.\"}", "split": "train"},
  {"id": "ficha_3", "instruccion": "Convertí el aviso en JSON con las claves tema y síntesis. Tema debe ser educación, movilidad o salud. No agregues información.", "entrada": "Se renovaron las becas para estudiantes del turno noche.", "respuesta": "{\"tema\": \"educación\", \"síntesis\": \"Se renovaron las becas para estudiantes del turno noche.\"}", "split": "train"},
  {"id": "ficha_4", "instruccion": "Convertí el aviso en JSON con las claves tema y síntesis. Tema debe ser educación, movilidad o salud. No agregues información.", "entrada": "La universidad invita a un seminario de historia regional.", "respuesta": "{\"tema\": \"educación\", \"síntesis\": \"La universidad invita a un seminario de historia regional.\"}", "split": "train"},
  {"id": "ficha_5", "instruccion": "Convertí el aviso en JSON con las claves tema y síntesis. Tema debe ser educación, movilidad o salud. No agregues información.", "entrada": "Docentes y estudiantes preparan una feria de libros.", "respuesta": "{\"tema\": \"educación\", \"síntesis\": \"Docentes y estudiantes preparan una feria de libros.\"}", "split": "train"},
  {"id": "ficha_6", "instruccion": "Convertí el aviso en JSON con las claves tema y síntesis. Tema debe ser educación, movilidad o salud. No agregues información.", "entrada": "El curso de programación tendrá encuentros semanales.", "respuesta": "{\"tema\": \"educación\", \"síntesis\": \"El curso de programación tendrá encuentros semanales.\"}", "split": "val"},
  {"id": "ficha_7", "instruccion": "Convertí el aviso en JSON con las claves tema y síntesis. Tema debe ser educación, movilidad o salud. No agregues información.", "entrada": "El centro cultural abre un taller de fotografía.", "respuesta": "{\"tema\": \"educación\", \"síntesis\": \"El centro cultural abre un taller de fotografía.\"}", "split": "val"},
  {"id": "ficha_8", "instruccion": "Convertí el aviso en JSON con las claves tema y síntesis. Tema debe ser educación, movilidad o salud. No agregues información.", "entrada": "La biblioteca ofrece encuentros de lectura para adolescentes.", "respuesta": "{\"tema\": \"educación\", \"síntesis\": \"La biblioteca ofrece encuentros de lectura para adolescentes.\"}", "split": "val"},
  {"id": "ficha_9", "instruccion": "Convertí el aviso en JSON con las claves tema y síntesis. Tema debe ser educación, movilidad o salud. No agregues información.", "entrada": "La escuela publicó su calendario de evaluaciones.", "respuesta": "{\"tema\": \"educación\", \"síntesis\": \"La escuela publicó su calendario de evaluaciones.\"}", "split": "test"},
  {"id": "ficha_10", "instruccion": "Convertí el aviso en JSON con las claves tema y síntesis. Tema debe ser educación, movilidad o salud. No agregues información.", "entrada": "El instituto habilitó consultas sobre equivalencias.", "respuesta": "{\"tema\": \"educación\", \"síntesis\": \"El instituto habilitó consultas sobre equivalencias.\"}", "split": "test"},
  {"id": "ficha_11", "instruccion": "Convertí el aviso en JSON con las claves tema y síntesis. Tema debe ser educación, movilidad o salud. No agregues información.", "entrada": "La universidad organiza una jornada de orientación vocacional.", "respuesta": "{\"tema\": \"educación\", \"síntesis\": \"La universidad organiza una jornada de orientación vocacional.\"}", "split": "test"},
  {"id": "ficha_12", "instruccion": "Convertí el aviso en JSON con las claves tema y síntesis. Tema debe ser educación, movilidad o salud. No agregues información.", "entrada": "El colectivo cambia su recorrido por obras en la avenida.", "respuesta": "{\"tema\": \"movilidad\", \"síntesis\": \"El colectivo cambia su recorrido por obras en la avenida.\"}", "split": "train"},
  {"id": "ficha_13", "instruccion": "Convertí el aviso en JSON con las claves tema y síntesis. Tema debe ser educación, movilidad o salud. No agregues información.", "entrada": "La estación anunció horarios especiales para el tren.", "respuesta": "{\"tema\": \"movilidad\", \"síntesis\": \"La estación anunció horarios especiales para el tren.\"}", "split": "train"},
  {"id": "ficha_14", "instruccion": "Convertí el aviso en JSON con las claves tema y síntesis. Tema debe ser educación, movilidad o salud. No agregues información.", "entrada": "Se cortará el tránsito durante la reparación del puente.", "respuesta": "{\"tema\": \"movilidad\", \"síntesis\": \"Se cortará el tránsito durante la reparación del puente.\"}", "split": "train"},
  {"id": "ficha_15", "instruccion": "Convertí el aviso en JSON con las claves tema y síntesis. Tema debe ser educación, movilidad o salud. No agregues información.", "entrada": "El municipio instaló nuevas bicisendas en el barrio.", "respuesta": "{\"tema\": \"movilidad\", \"síntesis\": \"El municipio instaló nuevas bicisendas en el barrio.\"}", "split": "train"},
  {"id": "ficha_16", "instruccion": "Convertí el aviso en JSON con las claves tema y síntesis. Tema debe ser educación, movilidad o salud. No agregues información.", "entrada": "La terminal informó demoras en servicios interurbanos.", "respuesta": "{\"tema\": \"movilidad\", \"síntesis\": \"La terminal informó demoras en servicios interurbanos.\"}", "split": "train"},
  {"id": "ficha_17", "instruccion": "Convertí el aviso en JSON con las claves tema y síntesis. Tema debe ser educación, movilidad o salud. No agregues información.", "entrada": "Las paradas de la línea local se trasladan dos cuadras.", "respuesta": "{\"tema\": \"movilidad\", \"síntesis\": \"Las paradas de la línea local se trasladan dos cuadras.\"}", "split": "train"},
  {"id": "ficha_18", "instruccion": "Convertí el aviso en JSON con las claves tema y síntesis. Tema debe ser educación, movilidad o salud. No agregues información.", "entrada": "El tren agrega frecuencias durante el fin de semana.", "respuesta": "{\"tema\": \"movilidad\", \"síntesis\": \"El tren agrega frecuencias durante el fin de semana.\"}", "split": "val"},
  {"id": "ficha_19", "instruccion": "Convertí el aviso en JSON con las claves tema y síntesis. Tema debe ser educación, movilidad o salud. No agregues información.", "entrada": "Una avenida permanece cerrada por mantenimiento.", "respuesta": "{\"tema\": \"movilidad\", \"síntesis\": \"Una avenida permanece cerrada por mantenimiento.\"}", "split": "val"},
  {"id": "ficha_20", "instruccion": "Convertí el aviso en JSON con las claves tema y síntesis. Tema debe ser educación, movilidad o salud. No agregues información.", "entrada": "El barrio solicita mejorar la iluminación de las paradas.", "respuesta": "{\"tema\": \"movilidad\", \"síntesis\": \"El barrio solicita mejorar la iluminación de las paradas.\"}", "split": "val"},
  {"id": "ficha_21", "instruccion": "Convertí el aviso en JSON con las claves tema y síntesis. Tema debe ser educación, movilidad o salud. No agregues información.", "entrada": "La ciclovía tendrá un desvío temporal.", "respuesta": "{\"tema\": \"movilidad\", \"síntesis\": \"La ciclovía tendrá un desvío temporal.\"}", "split": "test"},
  {"id": "ficha_22", "instruccion": "Convertí el aviso en JSON con las claves tema y síntesis. Tema debe ser educación, movilidad o salud. No agregues información.", "entrada": "Los usuarios reclaman un servicio nocturno de colectivos.", "respuesta": "{\"tema\": \"movilidad\", \"síntesis\": \"Los usuarios reclaman un servicio nocturno de colectivos.\"}", "split": "test"},
  {"id": "ficha_23", "instruccion": "Convertí el aviso en JSON con las claves tema y síntesis. Tema debe ser educación, movilidad o salud. No agregues información.", "entrada": "La terminal habilitó un nuevo acceso peatonal.", "respuesta": "{\"tema\": \"movilidad\", \"síntesis\": \"La terminal habilitó un nuevo acceso peatonal.\"}", "split": "test"},
  {"id": "ficha_24", "instruccion": "Convertí el aviso en JSON con las claves tema y síntesis. Tema debe ser educación, movilidad o salud. No agregues información.", "entrada": "El centro de salud ofrece turnos para vacunación.", "respuesta": "{\"tema\": \"salud\", \"síntesis\": \"El centro de salud ofrece turnos para vacunación.\"}", "split": "train"},
  {"id": "ficha_25", "instruccion": "Convertí el aviso en JSON con las claves tema y síntesis. Tema debe ser educación, movilidad o salud. No agregues información.", "entrada": "El hospital organiza una campaña de prevención.", "respuesta": "{\"tema\": \"salud\", \"síntesis\": \"El hospital organiza una campaña de prevención.\"}", "split": "train"},
  {"id": "ficha_26", "instruccion": "Convertí el aviso en JSON con las claves tema y síntesis. Tema debe ser educación, movilidad o salud. No agregues información.", "entrada": "La salita amplió el horario de atención pediátrica.", "respuesta": "{\"tema\": \"salud\", \"síntesis\": \"La salita amplió el horario de atención pediátrica.\"}", "split": "train"},
  {"id": "ficha_27", "instruccion": "Convertí el aviso en JSON con las claves tema y síntesis. Tema debe ser educación, movilidad o salud. No agregues información.", "entrada": "El municipio habilitó controles de presión arterial.", "respuesta": "{\"tema\": \"salud\", \"síntesis\": \"El municipio habilitó controles de presión arterial.\"}", "split": "train"},
  {"id": "ficha_28", "instruccion": "Convertí el aviso en JSON con las claves tema y síntesis. Tema debe ser educación, movilidad o salud. No agregues información.", "entrada": "El hospital informa dónde solicitar estudios clínicos.", "respuesta": "{\"tema\": \"salud\", \"síntesis\": \"El hospital informa dónde solicitar estudios clínicos.\"}", "split": "train"},
  {"id": "ficha_29", "instruccion": "Convertí el aviso en JSON con las claves tema y síntesis. Tema debe ser educación, movilidad o salud. No agregues información.", "entrada": "El centro barrial invita a una jornada de salud comunitaria.", "respuesta": "{\"tema\": \"salud\", \"síntesis\": \"El centro barrial invita a una jornada de salud comunitaria.\"}", "split": "train"},
  {"id": "ficha_30", "instruccion": "Convertí el aviso en JSON con las claves tema y síntesis. Tema debe ser educación, movilidad o salud. No agregues información.", "entrada": "La campaña de vacunación llegará a las escuelas.", "respuesta": "{\"tema\": \"salud\", \"síntesis\": \"La campaña de vacunación llegará a las escuelas.\"}", "split": "val"},
  {"id": "ficha_31", "instruccion": "Convertí el aviso en JSON con las claves tema y síntesis. Tema debe ser educación, movilidad o salud. No agregues información.", "entrada": "El consultorio abre nuevos turnos para controles.", "respuesta": "{\"tema\": \"salud\", \"síntesis\": \"El consultorio abre nuevos turnos para controles.\"}", "split": "val"},
  {"id": "ficha_32", "instruccion": "Convertí el aviso en JSON con las claves tema y síntesis. Tema debe ser educación, movilidad o salud. No agregues información.", "entrada": "El equipo de salud dará una charla sobre prevención.", "respuesta": "{\"tema\": \"salud\", \"síntesis\": \"El equipo de salud dará una charla sobre prevención.\"}", "split": "val"},
  {"id": "ficha_33", "instruccion": "Convertí el aviso en JSON con las claves tema y síntesis. Tema debe ser educación, movilidad o salud. No agregues información.", "entrada": "El hospital informa un cambio en la entrega de resultados.", "respuesta": "{\"tema\": \"salud\", \"síntesis\": \"El hospital informa un cambio en la entrega de resultados.\"}", "split": "test"},
  {"id": "ficha_34", "instruccion": "Convertí el aviso en JSON con las claves tema y síntesis. Tema debe ser educación, movilidad o salud. No agregues información.", "entrada": "La salita recibe consultas sobre tratamientos.", "respuesta": "{\"tema\": \"salud\", \"síntesis\": \"La salita recibe consultas sobre tratamientos.\"}", "split": "test"},
  {"id": "ficha_35", "instruccion": "Convertí el aviso en JSON con las claves tema y síntesis. Tema debe ser educación, movilidad o salud. No agregues información.", "entrada": "Se habilitó una jornada de controles en la plaza.", "respuesta": "{\"tema\": \"salud\", \"síntesis\": \"Se habilitó una jornada de controles en la plaza.\"}", "split": "test"}
]
'''

In [ ]:
ejemplos = json.loads(DATOS_EMBEBIDOS)

particiones = {"train": [], "val": [], "test": []}
for ejemplo in ejemplos:
    particiones[ejemplo["split"]].append(ejemplo)

for nombre, lista in particiones.items():
    print(f"{nombre}: {len(lista)} ejemplos")

## 3. El modelo base y LoRA

El modelo es Llama 3.2 de 3 mil millones de parámetros, ya ajustado para seguir instrucciones, en versión de 4 bits. La **cuantización de 4 bits** guarda los pesos con menos precisión para que el modelo entre en la memoria de la GPU. Su licencia es la de Llama 3.2: revisá la ficha del modelo antes de cambiarlo por otro.

- `max_seq_length`: largo máximo, en tokens, de lo que el modelo lee y escribe. Más largo gasta más memoria.
- `dtype=None`: Unsloth elige la precisión de los números según la GPU.
- `load_in_4bit=True`: carga el modelo cuantizado. Con `False` ocuparía mucha más memoria.

In [ ]:
from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 2048

# La primera vez descarga el modelo. Devuelve el modelo y su tokenizador.
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

**LoRA** no modifica los pesos originales: agrega matrices chicas, el **adaptador**, y entrena solo esas. Es como dejar el libro como está y escribirle notas al margen: las notas pesan poco y se pueden sacar.

- `r=16`: tamaño (rango) de las matrices. Más grande puede aprender más y ocupa más memoria; se suele probar entre 8 y 64.
- `lora_alpha=16`: escala del efecto del adaptador. Igual a `r`, el efecto no se amplifica ni se reduce.
- `target_modules`: en qué capas se agregan los adaptadores. Acá, las de atención y las de la red intermedia.
- El resto (`lora_dropout`, `bias`, `use_gradient_checkpointing`, `random_state`) toma los valores habituales de Unsloth; no cambia lo que medimos.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)
model.print_trainable_parameters()

> **Para mirar.** La última línea dice cuántos parámetros se van a entrenar y qué porcentaje del total son. Un adaptador recién agregado todavía no cambia al modelo, así que medir ahora equivale a medir el modelo base.

## 4. Antes de ajustar

Primero medimos el modelo tal como está, con dos preguntas por separado: si la respuesta es un **JSON válido** con las dos claves (el formato) y si el **tema es correcto** (el contenido). `do_sample=False` hace que el modelo elija siempre el token más probable, así que repetir la medición da lo mismo.

In [ ]:
import pandas as pd


def armar_texto(instruccion: str, entrada: str, respuesta: str = "") -> str:
    """Arma el pedido con la plantilla fija.

    Args:
        instruccion: Lo que se le pide al modelo.
        entrada: El aviso a procesar.
        respuesta: La respuesta esperada; vacia cuando solo se consulta.

    Returns:
        El texto con instruccion, entrada y respuesta.
    """
    return (
        "A continuación hay una instrucción que describe una tarea, junto "
        "con una entrada. Escribí una respuesta que complete el pedido.\n\n"
        f"### Instrucción:\n{instruccion}\n\n"
        f"### Entrada:\n{entrada}\n\n"
        f"### Respuesta:\n{respuesta}"
    )


def pedir_respuesta(instruccion: str, entrada: str) -> str:
    """Le da el pedido al modelo y devuelve lo que escribe.

    Args:
        instruccion: Lo que se le pide al modelo.
        entrada: El aviso a procesar.

    Returns:
        La respuesta generada, sin el pedido.
    """
    pedido = armar_texto(instruccion, entrada)
    entradas = tokenizer([pedido], return_tensors="pt").to("cuda")
    salidas = model.generate(
        **entradas, max_new_tokens=100, do_sample=False, use_cache=True
    )
    texto = tokenizer.decode(salidas[0], skip_special_tokens=True)
    return texto.split("### Respuesta:")[-1].strip()

In [ ]:
def evaluar(lista_de_ejemplos: list[dict]) -> pd.DataFrame:
    """Pide una respuesta por ejemplo y revisa formato y contenido.

    Args:
        lista_de_ejemplos: Ejemplos de una particion.

    Returns:
        Tabla con una fila por ejemplo: su respuesta, si es un JSON
        valido con las dos claves y si el tema es correcto.
    """
    casos = []
    for ejemplo in lista_de_ejemplos:
        respuesta = pedir_respuesta(ejemplo["instruccion"], ejemplo["entrada"])
        try:
            contenido = json.loads(respuesta)
        except json.JSONDecodeError:
            contenido = {}
        if not isinstance(contenido, dict):
            contenido = {}
        esperado = json.loads(ejemplo["respuesta"])
        casos.append({
            "entrada": ejemplo["entrada"],
            "respuesta": respuesta,
            "json_valido": "tema" in contenido and "síntesis" in contenido,
            "tema_correcto": contenido.get("tema") == esperado["tema"],
        })
    return pd.DataFrame(casos)


# for_inference prepara el modelo para generar texto; no sirve para entrenar.
FastLanguageModel.for_inference(model)
antes = evaluar(particiones["val"])
pd.set_option("display.max_colwidth", 120)
antes.head(4)

> **Para mirar.** Si el modelo agrega explicaciones, inventa claves o escribe el JSON entre comillas triples. Eso es lo que el ajuste va a intentar corregir.

## 5. Entrenar con SFTTrainer

El **ajuste fino supervisado** (SFT) sigue entrenando el modelo con pares de pedido y respuesta correcta. Cada ejemplo de `train` se convierte en un texto con la plantilla, **con la respuesta completa y el token de fin (`EOS_TOKEN`) al final**: sin esa marca, el modelo no aprende dónde termina y puede seguir escribiendo.

- `per_device_train_batch_size=2` y `gradient_accumulation_steps=2`: cada actualización de pesos mira 4 ejemplos.
- `num_train_epochs=4`: cuántas pasadas completas se dan por `train`.
- `learning_rate=2e-4`: tamaño de cada actualización. Es mayor que en un ajuste completo porque LoRA entrena pocas matrices.
- `optim="adamw_8bit"`: optimizador que gasta menos memoria.

In [ ]:
from datasets import Dataset

EOS_TOKEN = tokenizer.eos_token

textos_de_entrenamiento = []
for ejemplo in particiones["train"]:
    texto = armar_texto(
        ejemplo["instruccion"], ejemplo["entrada"], ejemplo["respuesta"]
    )
    textos_de_entrenamiento.append({"text": texto + EOS_TOKEN})

datos_de_entrenamiento = Dataset.from_list(textos_de_entrenamiento)

In [ ]:
from trl import SFTConfig, SFTTrainer
from unsloth import is_bfloat16_supported

# Se vuelve al modo de entrenamiento (antes estaba el de generacion).
FastLanguageModel.for_training(model)

# Si SFTTrainer rechaza processing_class, la version de TRL es otra: probar
# tokenizer=tokenizer (el cuaderno oficial de Unsloth) o revisar SFTConfig.
entrenador = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=datos_de_entrenamiento,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=2,
        warmup_steps=3,
        num_train_epochs=4,
        learning_rate=2e-4,
        # Segun la GPU se usa bfloat16 o float16: dos formas de guardar
        # numeros con menos memoria.
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        seed=3407,
        output_dir="outputs",
        report_to="none",
    ),
)
resultado = entrenador.train()

> **Para mirar.** La tabla que imprimió el entrenamiento: si la pérdida baja de un paso al siguiente, el modelo está aprendiendo el formato. Si queda quieta o sube, revisá `learning_rate`.

## 6. Después de ajustar y guardar

Se repite exactamente la medición anterior, con los mismos ejemplos de `val`. Después se guarda solo el **adaptador** (unos pocos MB), no el modelo completo.

In [ ]:
FastLanguageModel.for_inference(model)
despues = evaluar(particiones["val"])

pd.DataFrame({
    "momento": ["antes", "despues"],
    "json_valido": [antes["json_valido"].sum(), despues["json_valido"].sum()],
    "tema_correcto": [
        antes["tema_correcto"].sum(), despues["tema_correcto"].sum()
    ],
    "de": [len(antes), len(despues)],
})

> **Para mirar.** Hay solo 9 ejemplos de `val`: una o dos respuestas de diferencia pueden ser azar. Mirá por separado el formato (`json_valido`) y el contenido (`tema_correcto`): aprender un formato es mucho más fácil que aprender a decidir bien. Si ajustás algo después de abrir `test`, deja de ser una comprobación independiente.

In [ ]:
import shutil

CARPETA_DEL_ADAPTADOR = "adaptador_avisos_json"
model.save_pretrained(CARPETA_DEL_ADAPTADOR)
tokenizer.save_pretrained(CARPETA_DEL_ADAPTADOR)
archivo_zip = shutil.make_archive(
    CARPETA_DEL_ADAPTADOR, "zip", CARPETA_DEL_ADAPTADOR
)

# files.download solo existe dentro de Colab; fuera de Colab queda el .zip.
try:
    from google.colab import files
    files.download(archivo_zip)
except ImportError:
    print(f"Adaptador guardado en {archivo_zip}")

## 7. Otras vías

Este cuaderno es un camino entre varios para el mismo problema:

| Vía | Qué ofrece |
|---|---|
| `Trainer` de Hugging Face y PEFT | El camino general: sirve para casi cualquier modelo y no depende de una biblioteca en particular. Es lo que usamos en los cuadernos 02 y 03, con LoRA en lugar de ajustar todo. |
| Unsloth | Un camino de ajuste eficiente: el que usó este cuaderno. |
| [Unsloth Studio](https://unsloth.ai/docs/new/studio) | Una interfaz visual, de código abierto y en versión beta, para explorar y entrenar modelos sin escribir código. Sirve para probar ideas; no reemplaza entender los datos. |

Unsloth también puede exportar el modelo a GGUF para correrlo en una computadora propia; eso queda fuera de este cuaderno.

## Qué te llevás

- LoRA entrena un adaptador chico y deja intacto el modelo grande: por eso entra en una GPU de Colab.
- Formato y contenido se miden por separado: un JSON válido no garantiza un tema correcto.
- Ninguna herramienta reemplaza entender los datos, la evaluación y los límites. Con 9 ejemplos de `val`, una diferencia de uno o dos casos no alcanza para concluir.

**Para seguir:** elegí un aviso que el modelo ajustado resuelva mal, escribí una hipótesis (¿faltan ejemplos de ese tipo, la instrucción es ambigua o la categoría es dudosa?), cambiá una sola cosa, como `num_train_epochs` o `r`, y volvé a medir.